In [ ]:
from pathlib import Path
import os

# Set PROJECT_DATA_DIR before launching Jupyter to use data stored elsewhere.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".gitignore").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = Path(os.environ.get("PROJECT_DATA_DIR", str(PROJECT_ROOT / "data"))).expanduser()


## Anomaly Detection IEEE CIS Dataset

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest

Load Dataset

In [ ]:
df = pd.read_csv(str(DATA_DIR / 'train_merged_submission.csv'), low_memory=False)
print(df.shape)

In [ ]:
from sklearn.model_selection import train_test_split

TARGET_COL = "isFraud"


# First split: train vs temp
train_df, temp_df = train_test_split(
    df_model,
    test_size=0.20,
    stratify=df_model[TARGET_COL],
    random_state=42
)

# Second split: validation vs test
valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df[TARGET_COL],
    random_state=42
)

print("Train shape:", train_df.shape)
print("Valid shape:", valid_df.shape)
print("Test shape:", test_df.shape)

In [ ]:
def fraud_summary(name, frame, target_col):
    total = len(frame)
    frauds = int(frame[target_col].sum())
    rate = frauds / total
    print(f"{name:>5} | rows={total:,} | frauds={frauds:,} | fraud_rate={rate:.6%}")

fraud_summary("full", df_model, TARGET_COL)
fraud_summary("train", train_df, TARGET_COL)
fraud_summary("valid", valid_df, TARGET_COL)
fraud_summary("test",  test_df, TARGET_COL)

In [ ]:
X_train = train_df.drop(columns=[TARGET_COL]).copy()
y_train = train_df[TARGET_COL].astype(int)

X_valid = valid_df.drop(columns=[TARGET_COL]).copy()
y_valid = valid_df[TARGET_COL].astype(int)

X_test = test_df.drop(columns=[TARGET_COL]).copy()
y_test = test_df[TARGET_COL].astype(int)

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_valid:", X_valid.shape, "| y_valid:", y_valid.shape)
print("X_test :", X_test.shape,  "| y_test :", y_test.shape)

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

object_cols = X_train.select_dtypes(include=["object"]).columns.tolist() # find catgorical columns

# Make all object columns consistently strings
for df_part in [X_train, X_valid, X_test]:
    df_part[object_cols] = df_part[object_cols].astype(str)


#map categorical values to integers
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_train[object_cols] = encoder.fit_transform(X_train[object_cols])
X_valid[object_cols] = encoder.transform(X_valid[object_cols])
X_test[object_cols] = encoder.transform(X_test[object_cols])

print("Encoded object columns:", len(object_cols))

In [ ]:
print("X_valid shape:", X_valid.shape)
print("X_test shape :", X_test.shape)
print("y_valid shape:", y_valid.shape)
print("y_test shape :", y_test.shape)

iso = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

#iso.fit(X_train)  # Fit on all data (including frauds)
iso.fit(X_train[y_train == 0]) # Fit only on non-fraud data

valid_scores = -iso.decision_function(X_valid)
test_scores = -iso.decision_function(X_test)

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

valid_roc = roc_auc_score(y_valid, valid_scores)
valid_pr = average_precision_score(y_valid, valid_scores)

test_roc = roc_auc_score(y_test, test_scores)
test_pr = average_precision_score(y_test, test_scores)


#If I randomly choose one fraud transaction and one non-fraud transaction, 
# the model gives the fraud a higher anomaly score about 76.7% of the time. That is a ranking interpretation. It does not mean 76.7% accuracy.

print(f"Validation ROC-AUC: {valid_roc:.4f}")

print(f"Validation PR-AUC : {valid_pr:.4f}")

print(f"Test ROC-AUC      : {test_roc:.4f}")

print(f"Test PR-AUC       : {test_pr:.4f}")
# the PR-AUC shows how concentrated fraud is among the highest-risk alerts

In [ ]:
def frauds_in_top_k(scores, y_true, k=100):
    order = np.argsort(-scores)[:k]
    frauds = int(y_true.iloc[order].sum())
    rate = frauds / k
    return frauds, rate

for k in [50, 100, 500, 1000]:
    frauds, rate = frauds_in_top_k(test_scores, y_test, k)
    print(f"Top-{k}: frauds={frauds}, fraud_rate={rate:.4%}")


# The dataset fraud rate is 2.4%, but in the top 1000 anomaly-ranked transactions it rises to 9.5%.

## Local Outlier Factor

In [ ]:
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import roc_auc_score, average_precision_score

In [ ]:
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination="auto",
    novelty=True,
    n_jobs=-1
)

lof.fit(X_train[y_train == 0])

In [ ]:
valid_scores_lof = -lof.decision_function(X_valid)
test_scores_lof = -lof.decision_function(X_test)

print("valid_scores_lof shape:", valid_scores_lof.shape)
print("test_scores_lof shape :", test_scores_lof.shape)

In [ ]:
print(f"Validation ROC-AUC: {roc_auc_score(y_valid, valid_scores_lof):.4f}")
print(f"Validation PR-AUC : {average_precision_score(y_valid, valid_scores_lof):.4f}")
print(f"Test ROC-AUC      : {roc_auc_score(y_test, test_scores_lof):.4f}")
print(f"Test PR-AUC       : {average_precision_score(y_test, test_scores_lof):.4f}")

In [ ]:
def frauds_in_top_k(scores, y_true, k=100):
    order = np.argsort(-scores)[:k]
    frauds = int(y_true.iloc[order].sum())
    rate = frauds / k
    return frauds, rate

for k in [50, 100, 500, 1000]:
    frauds, rate = frauds_in_top_k(test_scores_lof, y_test, k)
    print(f"Top-{k:4d}: frauds found = {frauds:4d} | fraud rate = {rate:.4%}")

## One-Class SVM

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape, X_valid_scaled.shape, X_test_scaled.shape)

In [ ]:
normal_idx = (y_train == 0)
X_train_normal = X_train_scaled[normal_idx]

# sample for speed
sample_size = min(30000, X_train_normal.shape[0])
rng = np.random.RandomState(42)
sample_idx = rng.choice(X_train_normal.shape[0], size=sample_size, replace=False)

X_train_ocsvm = X_train_normal[sample_idx]

print("Training sample for One-Class SVM:", X_train_ocsvm.shape)

In [ ]:
from sklearn.svm import OneClassSVM

ocsvm = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.01 # expected fraud rate (1%)
)

ocsvm.fit(X_train_ocsvm)

In [ ]:
valid_scores_ocsvm = -ocsvm.decision_function(X_valid_scaled)
test_scores_ocsvm = -ocsvm.decision_function(X_test_scaled)

print("valid_scores_ocsvm shape:", valid_scores_ocsvm.shape)
print("test_scores_ocsvm shape :", test_scores_ocsvm.shape)

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

print(f"Validation ROC-AUC: {roc_auc_score(y_valid, valid_scores_ocsvm):.4f}")
print(f"Validation PR-AUC : {average_precision_score(y_valid, valid_scores_ocsvm):.4f}")
print(f"Test ROC-AUC      : {roc_auc_score(y_test, test_scores_ocsvm):.4f}")
print(f"Test PR-AUC       : {average_precision_score(y_test, test_scores_ocsvm):.4f}")

In [ ]:
def frauds_in_top_k(scores, y_true, k=100):
    order = np.argsort(-scores)[:k]
    frauds = int(y_true.iloc[order].sum())
    rate = frauds / k
    return frauds, rate

for k in [50, 100, 500, 1000]:
    frauds, rate = frauds_in_top_k(test_scores_ocsvm, y_test, k)
    print(f"Top-{k:4d}: frauds found = {frauds:4d} | fraud rate = {rate:.4%}")